**Author**: Ortega Rivera Leonardo Fabyan

<div style="margin: 0 auto; width: 80%; text-align: justify; padding: 20px; font-style: italic;">

**Abstract.** This work presents an exploratory analysis of a synthetic dataset of strong gravitational 
lensing images with three classes: no substructure, spherical substructure, and vortex substructure. 
While the dataset exhibits ideal structural properties 
(perfect class balance and uniform spatial resolution), 
our analysis reveals that class discrimination is not achievable through 
low-level statistical descriptors or linear projections. 
Both principal component analysis and clustering validity metrics 
indicate a near-complete overlap of class distributions in feature space. 
These findings suggest that the underlying discriminative structure 
is highly non-linear and spatially localized, 
motivating the use of deep convolutional architectures 
capable of hierarchical feature extraction.

</div>

# Introduction

The precise nature of dark matter remains one of the most prominent open questions in modern astrophysics. Strong galaxy-galaxy lensing offers a unique observational window, as dark matter substructures within the primary lens halo perturb the lensed images. While deep learning models, particularly Convolutional Neural Networks (CNNs), have emerged as natural candidates for extracting these subtle perturbations , a rigorous assessment of the input data manifold is a crucial prerequisite. This work conducts a comprehensive exploratory data analysis (EDA) on a synthetic strong lensing dataset to evaluate spatial separability. We demonstrate that while the dataset exhibits ideal structural consistency, class discrimination is unattainable via low-level statistical features or linear projections, thereby formally justifying the need for deep hierarchical feature extraction.

In [1]:
from plotly import express as px

import numpy as np
import pandas as pd

import sys
import os

sys.path.append(os.path.abspath('../src'))

from dataprocessing.dataloader import build_dataloaders  # type: ignore

DIR: str = 'eda_images'

os.makedirs(DIR, exist_ok=True)

def format_fig(fig) -> str:
    title = fig.layout.title.text
    return DIR + '/' + title.replace('.', '').replace(' ', '_').lower() + '.png'

# Methodology

## Dataset description

The dataset was synthetically generated utilizing the $\texttt{lenstronomy}$ package. 
It comprises strong gravitational lensing images categorized into three distinct morphological classes: no substructure, subhalo (spherical) substructure, and vortex substructure.

In [2]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('No Substructure',
                    'SubHalo (Sphere)',
                    'Vortex'))

img = np.load('../dataset/train/no/1.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=1)

img = np.load('../dataset/train/sphere/2.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=2)

img = np.load('../dataset/train/vort/3.npy').squeeze()
sub_fig = px.imshow(img)
fig.add_trace(sub_fig.data[0], row=1, col=3)

fig.write_image(fr'{DIR}/classes.png',
    width=1048, height=450)
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{eda_images/classes.png}
Figure 1: Example of a No Substructure, a Sphere, and a Vortex image.
\end{figure}

## Class Representativeness

Class imbalance is a well-known source of bias in supervised learning. 
To assess this risk, 
we compute the relative frequency (RF) of each class in both training and validation splits. 
The results show a perfectly uniform distribution 
($\text{RF} \approx 0.33$) 
ensuring that no class-dependent bias is introduced during training. 
Consequently, no rebalancing strategies 
(e.g., class weighting or synthetic sampling) 
are required.

In [3]:
# add tag: hide_cell
train_loader, valid_loader = build_dataloaders(
    '../dataset', 64, 128, 4, drop_last_train=False
)

split: train
	class: no (label: 0), number of samples: 10000
	class: sphere (label: 1), number of samples: 10000
	class: vort (label: 2), number of samples: 10000
	total samples: 30000
split: val
	class: no (label: 0), number of samples: 2500
	class: sphere (label: 1), number of samples: 2500
	class: vort (label: 2), number of samples: 2500
	total samples: 7500


In [4]:
train_rf = [
    round(n_samples / sum(train_loader.dataset.n_samples_per_class), 2)
    for n_samples in train_loader.dataset.n_samples_per_class]
    
valid_rf = [
    round(n_samples / sum(valid_loader.dataset.n_samples_per_class), 2)
    for n_samples in valid_loader.dataset.n_samples_per_class]
    
df = pd.DataFrame({
    'Classes': valid_loader.dataset.labels_name,
    'RF train': train_rf,
    'RF valid': valid_rf
})
    
fig = px.bar(df, x='Classes', 
    y=['RF train', 'RF valid'],
    title='Figure 2. Relative Frequency (RF)',
    text_auto=True, barmode='group')
    
fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{eda_images/figure_2_relative_frequency_(rf).png}
Figure 2: Relative frequency of each class.
\end{figure}

## Spatial Consistency and Resolution

Variability in input geometric dimensions can introduce severe architectural constraints and noise in convolutional models. An exhaustive dimensional analysis confirms absolute structural homogeneity across the dataset: all samples invariably exhibit a spatial resolution of $150 \times 150$ pixels with a $1:1$ aspect ratio and a single grayscale channel. This structural invariance guarantees that no aggressive preprocessing - such as cropping, padding, or affine transformations - is required, thereby preserving the original spatial morphology of the lensing phenomena.

In [5]:
channels: list[int] = []
heights: list[int] = []
widths: list[int] = []
aspect_ratios: list[float] = []
labels_list: list[int] = []

for images, labels in train_loader:
    for i in range(images.shape[0]): # iterating over a batch
        channels.append(images[i].shape[0])
        heights.append(images[i].shape[1])
        widths.append(images[i].shape[2])
        aspect_ratios.append(heights[-1] / widths[-1])
        labels_list.append(
            valid_loader.dataset.labels_name[labels[i].item()])
        
df_sc_r = pd.DataFrame({
    'channel': channels,
    'height': heights,
    'width': widths,
    'aspect_ratio': aspect_ratios,
    'label': labels_list
})

# Spatial Separability Analysis

## Low-Level Statistical Overlap

First-order statistical descriptors (mean $\mu$ and standard deviation $\sigma$) were extracted to evaluate the discriminative capacity of global intensity patterns. As illustrated in Figure 3, the feature space exhibits a severe, near-perfect overlap across all three classes. This empirical evidence demonstrates that global luminance and contrast variations are entirely decoupled from the presence of dark matter substructures.

In [6]:
list_images: list[np.ndarray] = []
features = []

for images, labels in train_loader:
    list_images.append(images.reshape(len(images), -1))
    
    for i in range(images.shape[0]): 
        img = images[i, 0].numpy().flatten()
        feat = {}
        
        feat['mean'] = np.mean(img)
        feat['std'] = np.std(img)
        feat['label_id'] = labels[i].item()
        feat['label_name'] = valid_loader.dataset.labels_name[labels[i].item()]
        
        features.append(feat)
        
X = np.concatenate(list_images, axis=0)
df_features = pd.DataFrame(features)

fig = px.scatter(
    df_features,
    x='mean',
    y='std',
    color='label_name',
    title='Figure 3. Mean vs Std'
)
fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{eda_images/figure_3_mean_vs_std.png}
Figure 3: Mean and Standard Desviation of each class.
\end{figure}

## Linear Subspace Analysis and Clustering Metrics

To formally quantify the high-dimensional entanglement, we projected the data onto a linear subspace utilizing the top 25 Principal Components. The clustering validity metrics strongly corroborate the initial low-level findings:
The Silhouette Score yielded a value of $−0.0022$. This negative, near-zero magnitude mathematically confirms that samples frequently lie on or cross the decision boundaries of neighboring classes in the linear manifold.
The Davies-Bouldin Index reached an exceptionally high value of $109.92$, indicating an intra-class variance that overwhelmingly eclipses inter-class dispersion.
Finally, the Fisher Discriminant Ratio (FDR) across the principal components peaked at merely 
$\approx 800 \mu$ (Figure 4). Even at its maximum, these vanishingly small FDR values definitively prove that the variance induced by identical macroscopic lensing topologies completely masks the localized signal of the subhalos in any linear projection.

In [7]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=25)
X_pca = pca.fit_transform(X_scaled)

In [8]:
# add tag: hide_cell
from sklearn.metrics import silhouette_score

score = silhouette_score(X_pca, df_features['label_id'])
print(score)

-0.002263626316562295


In [9]:
# add tag: hide_cell
from sklearn.metrics import davies_bouldin_score

db_score = davies_bouldin_score(X_pca, df_features['label_id'])
print(db_score)

109.92109093070724


In [10]:
# add tag: hide_cell
def fisher_discriminant_ratio(X, y):
    classes = np.unique(y)
    
    mu = np.mean(X, axis=0) # media global
    
    S_B = np.zeros(X.shape[1])
    S_W = np.zeros(X.shape[1])
    
    for c in classes:
        X_c = X[y == c]
        mu_c = np.mean(X_c, axis=0)
        n_c = X_c.shape[0]
        
        # Between-class
        S_B += n_c * (mu_c - mu) ** 2
        
        # Within-class
        S_W += np.sum((X_c - mu_c) ** 2, axis=0)
    
    eps = 1e-8
    FDR = S_B / (S_W + eps)
    
    return FDR

fisher = fisher_discriminant_ratio(X_pca, df_features['label_id'])
print(fisher)

[8.75903065e-05 4.22333551e-04 2.64670988e-05 5.86154293e-06
 2.25287365e-05 2.90606696e-05 7.89355664e-04 1.69174307e-05
 1.48413144e-04 5.91744834e-05 4.38572583e-05 5.08104496e-04
 1.00301904e-05 3.84012299e-05 1.92659349e-05 2.48521253e-05
 3.55314415e-04 1.10923899e-05 3.29900389e-05 4.05836483e-05
 1.75674924e-05 1.34241041e-04 9.78507442e-06 4.03799741e-04
 3.68246911e-05]


In [11]:
fisher_df = pd.DataFrame({
    'Fisher Ratio': fisher,
    'Principal Component (1 to 25)': range(1, len(fisher) + 1)
})

fig = px.bar(fisher_df, 
    x='Principal Component (1 to 25)', 
    y='Fisher Ratio',
    title='Figure 4. Fisher Ratio per PCA component',
    color='Fisher Ratio')
fig.write_image(format_fig(fig))
fig.show()

\begin{figure}[h]
\centering
\includegraphics[width=1\textwidth]{eda_images/figure_4_fisher_ratio_per_pca_component.png}
Figure 4: Fisher Ratio per PCA component.
\end{figure}

# Conclusion

In conclusion, while the dark matter substructure dataset provides a highly controlled and balanced environment (eliminating the need for morphological preprocessing or synthetic resampling), the intrinsic signal of the substructures (spheres and vortices) is deeply embedded within the global image variance. The failure of linear separability metrics (Silhouette $\approx 0$, DB Index $>100$, and vanishing Fisher Ratios) strictly rules out shallow learning techniques. Consequently, this EDA formally establishes the baseline complexity of the dataset. By demonstrating the inadequacy of shallow learning techniques , it provides the rigorous mathematical justification required for the future deployment of spatially-aware deep learning architectures, such as Convolutional Neural Networks, to isolate the localized perturbations induced by dark matter subhalos.